# Synthea Data Generator 
***

Setup: 

* A catalog, schema, and volumne setup with write permissions for the principal executing this notebook. 
* A cluster with Java JDK version 17 set as the default.
* The **synthea-with-dependencies.jar** available for the cluster to use. 
* A preconfigured Synthea configuration file.  

All of these requirements can be set up using the notebooks found in the 00-setup-notebooks folder of this repo, or from the original on Github:  [https://github.com/matthew-gigl-db/db-nosql](https://github.com/matthew-gigl-db/db-nosql)

***

In [0]:
import subprocess

result = subprocess.run(["java", "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
java_version = result.stdout + result.stderr
print(java_version)

Typical output using default cluster JDK: 
> openjdk version "1.8.0_392"  
> OpenJDK Runtime Environment (Zulu 8.74.0.17-CA-linux64) (build 1.8.0_392-b08)  
> OpenJDK 64-Bit Server VM (Zulu 8.74.0.17-CA-linux64) (build 25.392-b08, mixed mode)  
>  

Output when the cluster JDK is set to version 17:  
> openjdk version "17.0.9" 2023-10-17 LTS  
> OpenJDK Runtime Environment Zulu17.46+19-CA (build 17.0.9+8-LTS)  
> OpenJDK 64-Bit Server VM Zulu17.46+19-CA (build 17.0.9+8-LTS, mixed mode, sharing)  
>  

In [0]:
if java_version.split('openjdk version "')[1].startswith("17"):
  print("Java Version is set correctly with version 17+")
else: 
  raise Exception("Error: Please ensure that java version 17 is set as the cluster default.  Please see https://docs.databricks.com/en/dev-tools/sdk-java.html#create-a-cluster-that-uses-jdk-17 for more information.")

***
### Set Catalog, Schema, and Volume Paths 

In [0]:
dbutils.widgets.text(name = "catalog_use", defaultValue="", label="Catalog Name")
dbutils.widgets.text(name = "schema_use", defaultValue="synthea", label="Schema Name")

In [0]:
catalog_name = dbutils.widgets.get(name = "catalog_use")
schema_name = dbutils.widgets.get(name = "schema_use")
volume_path = f"/Volumes/{catalog_name}/{schema_name}/synthetic_files_raw/"
print(f"""
  catalog_name = {catalog_name}
  schema_name = {schema_name}
  volume_path = {volume_path}
""")

***
### Set Min and Max Values for Random Number of Patient Records Generated 

Each run of this notebook will generate a random number of Synthea Patient records between the minimum number and the maximum number set.  This helps us simulate the variability in how patient records might arrive in a real world environment.  

In [0]:
from datetime import date

dbutils.widgets.text(name = "min_records", defaultValue="1", label = "Minimum Generated Record Count")
dbutils.widgets.text(name = "max_records", defaultValue="1000", label = "Maximum Generated Record Count")
# end_date caps the simulation timeline to prevent "Person does not have insurance" errors from the CCDA exporter
# Defaults to Dec 31 of the previous year to stay within Synthea's payer data coverage
dbutils.widgets.text(name = "end_date", defaultValue = f"{date.today().year - 1}1231", label = "Simulation End Date")

In [0]:
# Check if "min_records" is an integer
try:
  min_records = int(dbutils.widgets.get("min_records"))
except ValueError:
  raise Exception("Please set the minimum generated record count to an integer value")
min_records

In [0]:
# Check if "max_records" is an integer
try:
  max_records = int(dbutils.widgets.get("max_records"))
except ValueError:
  raise Exception("Please set the maximum generated record count to an integer value")
max_records

*** 
### Data Generation Function 

In [0]:
from random import randint

In [0]:
def data_generator(volume_path: str = volume_path, config_file_path: str = f"{volume_path}synthea_config.txt", min_record_cnt: int = min_records, max_record_cnt: int = max_records, additional_options: str = "", verbose: bool = False):
  random_record_count = randint(min_records, max_records)
  # FHIR/CCDA use atomic single-file writes that work fine on UC Volumes FUSE.
  # CSV uses long-lived buffered writers that don't flush through FUSE on serverless.
  # Solution: write to local /tmp with symlinks for FHIR/CCDA pointing to the volume,
  # so only CSV actually lands on local disk and needs copying.
  local_output = "/tmp/synthea_output"
  volume_output = f"{volume_path}output"
  command = (
  f"""rm -rf {local_output} && mkdir -p {local_output}
  mkdir -p {volume_output}/fhir {volume_output}/ccda {volume_output}/metadata
  ln -sf {volume_output}/fhir {local_output}/fhir
  ln -sf {volume_output}/ccda {local_output}/ccda
  ln -sf {volume_output}/metadata {local_output}/metadata
  cd {volume_path}
  java -Dlog4j2.disable.jmx=true -jar synthea-with-dependencies.jar -c {config_file_path} -p {random_record_count} --exporter.baseDirectory={local_output}/ {additional_options}
  cp -r {local_output}/csv {volume_output}/ 2>/dev/null
  rm -rf {local_output}
  """)
  if verbose == True:
    print(command)
  result = subprocess.run([command], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, shell=True)
  return result

***
### Generate Records

In [0]:
end_date_raw = dbutils.widgets.get("end_date").replace("-", "").strip()
end_date = end_date_raw if end_date_raw else date.today().strftime("%Y%m%d")

run_results = data_generator(
  volume_path=volume_path
  ,config_file_path=f"{volume_path}synthea_config.txt"
  ,min_record_cnt=min_records
  ,max_record_cnt=max_records
  ,additional_options=f"-e {end_date}"
  ,verbose=True
)

In [0]:
print(run_results.stderr)

In [0]:
print(run_results.stdout)